# Semana 32 · ¿De qué están hechas las estrellas?

Este cuaderno acompaña a la sesión de proyecto de la semana 32: los niveles de energía del hidrógeno, el ajuste de la serie de Balmer y la lectura de un espectro.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. La escalera de energía del hidrógeno

$$E_n = -\frac{13{,}6\ \text{eV}}{n^2}.$$

Dibujamos los niveles y los saltos de tres series. La longitud de onda de cada salto sale de $\lambda\,(\text{nm}) = 1240/\Delta E\,(\text{eV})$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def E(n):
    return -13.6 / n**2

fig, ax = plt.subplots(figsize=(10, 6))
for n in range(1, 8):
    ax.hlines(E(n), 0, 10, color="gray", lw=1)
    if n <= 4:
        ax.text(10.1, E(n), f"n = {n}", va="center", fontsize=8)
ax.hlines(0, 0, 10, color="black", lw=1, ls="--"); ax.text(10.1, 0.3, "ionizado", fontsize=8)

series = [("Lyman (UV)", 1, "darkviolet", 0.5), ("Balmer (visible)", 2, "crimson", 4.0), ("Paschen (IR)", 3, "darkorange", 7.5)]
for nombre, nf, color, x0 in series:
    for i, ni in enumerate(range(nf + 1, nf + 5)):
        x = x0 + 0.5 * i
        ax.annotate("", xy=(x, E(nf)), xytext=(x, E(ni)), arrowprops=dict(arrowstyle="->", color=color))
        lam = 1240 / (E(ni) - E(nf))
        if i == 0:
            ax.text(x - 0.15, (E(ni) + E(nf)) / 2, f"{lam:.0f} nm", fontsize=7, ha="right", color=color)
    ax.text(x0 + 0.7, E(nf) - 0.7, nombre, color=color, ha="center", fontsize=9)
ax.set_ylabel("energía (eV)"); ax.set_xticks([]); ax.set_ylim(-14.6, 1.2)
ax.set_title("Niveles de energía del hidrógeno y tres series de líneas")
plt.show()

## 2. Ajustar la serie de Balmer

Estas son las longitudes de onda **medidas** de las primeras líneas de Balmer, en el aire (los valores de las tablas de laboratorio, como las del NIST). Comprobamos que $1/\lambda$ frente a $1/n^2$ es una recta y medimos la constante de Rydberg con su pendiente.

In [ ]:
n = np.array([3, 4, 5, 6, 7, 8, 9])
lam_nm = np.array([656.279, 486.135, 434.047, 410.175, 397.007, 388.905, 383.539])

x = 1 / n**2
y = 1 / (lam_nm * 1e-9)               # en m⁻¹
pendiente, ordenada = np.polyfit(x, y, 1)
R = -pendiente
print(f"R (pendiente)        = {R:.5e} m⁻¹")
print(f"R (ordenada × 4)     = {4 * ordenada:.5e} m⁻¹")

residuos = y - (pendiente * x + ordenada)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
a1.plot(x, y, "o"); a1.plot(x, pendiente * x + ordenada, "-")
a1.set_ylabel("1/λ (m⁻¹)"); a1.set_title("Serie de Balmer linealizada"); a1.grid(alpha=0.3)
a2.axhline(0, color="gray"); a2.plot(x, residuos, "o")
a2.set_xlabel("1/n²"); a2.set_ylabel("residuo (m⁻¹)"); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"residuo relativo máximo: {np.abs(residuos / y).max():.1e}")

Los residuos son de unas pocas partes por millón, sin una forma clara: la ley se cumple casi perfectamente.

El valor que sale, unos $1{,}0971\times10^7$ m⁻¹, es un poco mayor que el de los libros para el hidrógeno ($1{,}09678\times10^7$ m⁻¹). La diferencia es real y tiene explicación: las longitudes de onda están medidas **en el aire**, donde la luz va un 0,03 % más despacio que en el vacío y las ondas son un 0,03 % más cortas. Compruébalo:

In [ ]:
n_aire = 1.000277
print(f"R corregido al vacío: {R / n_aire:.5e} m⁻¹")

## 3. Leer un espectro

Ahora generamos un espectro **sintético** de una estrella parecida al Sol. Las posiciones de las líneas son las reales; sus profundidades y anchuras, y el ruido, son inventados para el ejercicio. Así sabemos la respuesta y podemos comprobar si nuestro método la encuentra.

In [ ]:
rng = np.random.default_rng(32)
h, c, kB = 6.626e-34, 2.998e8, 1.381e-23

def planck(lam_nm, T):
    lam = lam_nm * 1e-9
    return 1 / lam**5 / np.expm1(h * c / (lam * kB * T))

lineas = {  # longitud de onda (nm): (elemento, profundidad, anchura σ en nm) — profundidades inventadas
    393.37: ("Ca II K", 0.85, 0.50), 396.85: ("Ca II H", 0.80, 0.45), 410.17: ("H-delta", 0.45, 0.30),
    434.05: ("H-gamma", 0.50, 0.30), 486.13: ("H-beta", 0.55, 0.30), 516.73: ("Mg b", 0.40, 0.12),
    517.27: ("Mg b", 0.45, 0.12), 518.36: ("Mg b", 0.50, 0.12), 527.04: ("Fe", 0.35, 0.08),
    588.99: ("Na D2", 0.65, 0.10), 589.59: ("Na D1", 0.60, 0.10), 656.28: ("H-alfa", 0.60, 0.30),
}
lam = np.arange(380, 700, 0.05)
continuo = planck(lam, 5772); continuo /= continuo.max()
perfil = np.ones_like(lam)
for l0, (_, prof, s) in lineas.items():
    perfil *= 1 - prof * np.exp(-0.5 * ((lam - l0) / s) ** 2)
flujo = continuo * perfil + rng.normal(0, 0.006, lam.size)

plt.figure(figsize=(12, 4))
plt.plot(lam, flujo, lw=0.6)
plt.xlabel("longitud de onda (nm)"); plt.ylabel("flujo (unidades arbitrarias)")
plt.title("Espectro sintético de una estrella como el Sol"); plt.grid(alpha=0.3); plt.show()

**Paso 1: estimar el continuo.** No sabemos (en principio) cuál es el continuo. Una forma sencilla de estimarlo: ajustar un polinomio suave al espectro, quitar los puntos que quedan muy por debajo (las líneas) y repetir unas cuantas veces.

In [ ]:
usar = np.ones_like(lam, dtype=bool)
for vuelta in range(8):
    coef = np.polyfit(lam[usar], flujo[usar], 5)
    estimado = np.polyval(coef, lam)
    usar = flujo > estimado - 0.01           # descartamos lo que queda por debajo
normalizado = flujo / estimado

fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
a1.plot(lam, flujo, lw=0.6); a1.plot(lam, estimado, "r", lw=1.5, label="continuo estimado"); a1.legend()
a2.plot(lam, normalizado, lw=0.6); a2.axhline(1, color="r", lw=0.8)
a2.set_xlabel("longitud de onda (nm)"); a2.set_ylabel("flujo normalizado")
plt.tight_layout(); plt.show()

**Paso 2: encontrar y medir las líneas.** Buscamos mínimos locales que bajen de 0,8, y medimos su profundidad y su anchura a media profundidad. Para no confundir el ruido con líneas, suavizamos antes el espectro con una media móvil.

In [ ]:
suave = np.convolve(normalizado, np.ones(5) / 5, mode="same")
encontradas = []
for i in range(5, lam.size - 5):
    if suave[i] < 0.8 and suave[i] == suave[i - 5:i + 6].min():
        prof = 1 - suave[i]
        mitad = 1 - prof / 2
        izq = i
        while izq > 0 and suave[izq] < mitad: izq -= 1
        der = i
        while der < lam.size - 1 and suave[der] < mitad: der += 1
        encontradas.append((lam[i], prof, lam[der] - lam[izq]))

def identificar(l):
    l0 = min(lineas, key=lambda k: abs(k - l))
    return lineas[l0][0] if abs(l0 - l) < 0.3 else "?"

print(f"{'posición (nm)':>14}{'profundidad':>13}{'anchura (nm)':>14}   identificación")
for l, p, a in encontradas:
    print(f"{l:>14.2f}{p:>13.2f}{a:>14.2f}   {identificar(l)}")

Fíjate en que las profundidades medidas no coinciden exactamente con las que pusimos: el suavizado y el ruido las cambian, y algunas líneas cercanas se mezclan. Es lo que pasa con los espectros reales, y por eso los astrónomos ajustan modelos de perfil a cada línea en lugar de mirar solo el mínimo.

## 4. Un espectro real (opcional, sin probar)

El *Sloan Digital Sky Survey* permite descargar espectros reales. Desde su herramienta de exploración (*SkyServer Explore*), buscando un objeto con espectro, se puede descargar el espectro como archivo CSV. Si lo subes a Colab (icono de la carpeta, a la izquierda, y luego «Subir»), la celda siguiente intenta leerlo. **Esta parte no está probada**: los nombres de las columnas pueden cambiar entre versiones del SDSS. Ten en cuenta también que el SDSS da las longitudes de onda en ångströms y en el vacío, y que las galaxias y los cuásares tienen el espectro desplazado al rojo (semana 41).

In [ ]:
import os
archivo = "espectro_sdss.csv"          # cambia el nombre por el del archivo que hayas subido
if os.path.exists(archivo):
    datos = np.genfromtxt(archivo, delimiter=",", names=True)
    print("columnas:", datos.dtype.names)
    col_lam, col_flujo = datos.dtype.names[0], datos.dtype.names[1]
    plt.figure(figsize=(12, 4))
    plt.plot(datos[col_lam] / 10, datos[col_flujo], lw=0.5)   # Å → nm
    plt.xlabel("longitud de onda (nm, vacío)"); plt.ylabel(col_flujo); plt.show()
else:
    print(f"No encuentro {archivo}. Sube un espectro en CSV para probar esta parte.")

## 5. Experimenta

1. Desplaza todo el espectro sintético 0,2 nm hacia el rojo (como si la estrella se alejara) y comprueba qué velocidad sale con $v = c\,\Delta\lambda/\lambda$ para cada línea.
2. Aumenta el ruido a 0,03. ¿Qué líneas dejas de detectar?
3. Haz el ajuste de Balmer solo con las tres primeras líneas. ¿Cuánto cambia $R$?

In [ ]:
# Tu experimento aquí